In [0]:
dbutils.widgets.text("p_batch_id", "", "Batch ID")

v_batch_id = dbutils.widgets.get("p_batch_id").strip()

if not v_batch_id:
    raise ValueError("Enter a batch ID")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
bronze_path = "s3://bizf1project123/Bronze/Aaisha/incremental-load/constructors"

silver_path = "s3://bizf1project123/Silver/Aaisha/incremental-load/constructors"

In [0]:
constructors_df = (
    spark.read
        .format("delta")
        .load(bronze_path)
        .filter(F.col("batch_id") == v_batch_id)
)

In [0]:
display(constructors_df)

In [0]:
constructors_dropped_df = constructors_df.drop("url")

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
        .withColumnRenamed("constructorId", "constructor_id")
        .withColumnRenamed("name", "constructor_name")
)

In [0]:
display(constructors_renamed_df)

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(
    ["constructor_id"]
)

In [0]:
constructors_final_df = constructors_distinct_df.withColumn(
    "nationality",
    F.initcap(F.col("nationality"))
)

In [0]:
display(constructors_final_df)

In [0]:
constructors_final_df = (
    constructors_final_df
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

In [0]:
if not DeltaTable.isDeltaTable(spark, silver_path):

    (
        constructors_final_df.write
            .format("delta")
            .mode("overwrite")
            .save(silver_path)
    )

else:

    silver_delta = DeltaTable.forPath(spark, silver_path)

    (
        silver_delta.alias("t")
            .merge(
                constructors_final_df.alias("s"),
                "t.constructor_id = s.constructor_id"
            )
            .whenMatchedUpdate(
                condition="s.batch_id >= t.batch_id",
                set={
                    "constructor_name": "s.constructor_name",
                    "nationality": "s.nationality",
                    "ingestion_timestamp": "s.ingestion_timestamp",
                    "source_file": "s.source_file",
                    "batch_id": "s.batch_id",
                    "updated_timestamp": "s.updated_timestamp"
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
    )

In [0]:
constructors_silver_df = (
    spark.read
        .format("delta")
        .load(silver_path)
)

display(constructors_silver_df)